# get the data :

In [2]:
import os
import json
import itertools
import csv
from google.colab import drive


# Reconnecter Drive proprement
try:
    drive.flush_and_unmount()
except:
    pass
drive.mount('/content/drive', force_remount=True)


# --- CONFIGURATION ---
DATA_DIRS = [
    #"/content/drive/.shortcut-targets-by-id/14T7-Rrikc5faA88C46ODGJuh9gw_rH00/Abderahmane"
    # "/content/drive/.shortcut-targets-by-id/1p-Puanbb3bPpRFF5xKN2a4vLn-G7eupa/",
    # "/content/drive/.shortcut-targets-by-id/1GCz66KY_YAruf3NxHOOtMw5H3ykjQRN3/Bilel",
     "/content/drive/.shortcut-targets-by-id/1fwW853am4kHWdNXr3yuOe-q-P2LTDnOW/Rayane",
    # "/content/drive/.shortcut-targets-by-id/1Lh4jJesDGb9sWrUb0Hlg6a847oMsiPj4/Zohra"

]


for folder in DATA_DIRS:
    if not os.path.exists(folder):
        print(f"{folder} → Folder not found")
        continue

    count = 0
    for _, _, files in os.walk(folder):
        count += len(files)

    print(f"{folder} →  {count} files")

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1fwW853am4kHWdNXr3yuOe-q-P2LTDnOW/Rayane →  1415 files


# Filter

In [3]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 1.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=fbf502c932ab28c1c909e283e43354788ba6de373b725fa29a95f00fcc6d8bfc
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [4]:
from langdetect import detect, LangDetectException

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
try:
    import ujson as json_fast
except:
    json_fast = json
from string import ascii_letters

# ===============================
# CONFIG
# ===============================

OUTPUT_FILE = "/content/drive/MyDrive/NLP Data/Scopus_english/scopus_english_ids_rayane.txt"
PROCESSED_FILE_LOG = "/content/drive/MyDrive/NLP Data/processed_files_rayane.txt"
LOG_EVERY = 20   # write to log every N files (massive speed boost)

# ===============================
# FAST UTILITIES
# ===============================

def load_processed_files():
    if not os.path.exists(PROCESSED_FILE_LOG):
        return set()
    with open(PROCESSED_FILE_LOG, "r", encoding="utf-8") as f:
        return set(line.strip() for line in f if line.strip())

def fast_is_english(text):
    """EXTREMELY FAST English detection using ASCII ratio."""
    if not text:
        return False

    text = text.strip()
    if len(text) < 4:
        return False

    letters = [c for c in text if c.isalpha()]
    if not letters:
        return False

    ascii_count = sum(c in ascii_letters for c in letters)
    ratio = ascii_count / len(letters)

    return ratio >= 0.90  # threshold for English

def reconstruct_abstract(inv):
    if not inv:
        return ""
    out = []
    for word, pos_list in inv.items():
        for pos in pos_list:
            out.append((pos, word))
    out.sort(key=lambda x: x[0])
    return " ".join(w for _, w in out)

def is_article_english(article):
    title = article.get("title") or article.get("display_name", "")
    if not fast_is_english(title):
        return False

    inv = article.get("abstract_inverted_index")
    if inv:
        abstract = reconstruct_abstract(inv)
        return fast_is_english(abstract)

    return True

def load_articles(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return json_fast.load(f)
    except:
        return []

def check_scopus(article):
    try:
        return article["primary_location"]["source"]["is_indexed_in_scopus"]
    except:
        return False

# ===============================
# MAIN LOOP
# ===============================

already = load_processed_files()
print(f"Loaded {len(already)} previously processed files.\n")

processed_buffer = []
processed_count = 0

with open(OUTPUT_FILE, "a", encoding="utf-8") as out:

    for data_dir in DATA_DIRS:
        print(f"Processing directory: {data_dir}\n")

        for root, dirs, files in os.walk(data_dir):
            for file in files:
                if not file.endswith(".json"):
                    continue

                path = os.path.join(root, file)

                # skip if already processed
                if path in already:
                    continue

                # -------- PROCESS FILE --------
                articles = load_articles(path)

                if isinstance(articles, dict) and "articles" in articles:
                    articles = articles["articles"]

                if isinstance(articles, list):
                    for article in articles:
                        if check_scopus(article) and is_article_english(article):
                            article_id = article.get("id")
                            if article_id:
                                out.write(article_id + "\n")

                processed_buffer.append(path)
                processed_count += 1

                # -------- REAL-TIME PROGRESS --------
                print(f"\rProcessed files: {processed_count}", end="")

                # Write log every LOG_EVERY files
                if processed_count % LOG_EVERY == 0:
                    with open(PROCESSED_FILE_LOG, "a", encoding="utf-8") as log:
                        log.write("\n".join(processed_buffer) + "\n")
                    processed_buffer.clear()
                    print(f"\n[+] Checkpoint: {processed_count} files logged.")

    # Write any remaining log entries
    if processed_buffer:
        with open(PROCESSED_FILE_LOG, "a", encoding="utf-8") as log:
            log.write("\n".join(processed_buffer) + "\n")

print("\n\n==============================")
print("       PROCESS COMPLETE")
print("==============================")
print(f"Total new files processed: {processed_count}")
print("==============================\n")


Loaded 40 previously processed files.

Processing directory: /content/drive/.shortcut-targets-by-id/1fwW853am4kHWdNXr3yuOe-q-P2LTDnOW/Rayane

Processed files: 6